# Caption relation supervision audit (CPU)

Enable Internet; Accelerator None. Fill SELECTED and PILOT_REVIEW with mounted paths. Attach original full V1 grounded cache and COCO caption JSON if absent. No images, model download, checkpoint or training. This counts lexical coverage only; missing words do not prove a false relation. The expected review input is decoder_relation_memory_adaptation_512_review.zip, or the extracted folder containing legacy/ and direct/. Return caption_relation_supervision_audit_review.zip.


In [ ]:
print("Caption/relation supervision audit started (CPU only).", flush=True)
import subprocess, sys
from pathlib import Path

SELECTED = Path("/kaggle/input/datasets/ducanh2403/CHANGE_ME/selected_relations_full100.json")
# A review ZIP, or its extracted directory containing legacy/ and direct/.
PILOT_REVIEW = Path("/kaggle/input/datasets/ducanh2403/CHANGE_ME_REVIEW")
COCO_JSON = Path("/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json")
TRAIN_GROUNDED_CACHE = Path("/kaggle/input/notebooks/ducanh2403/build-grounded-relation-v1/grounded_relation_v1_full/grounded_relation_cache.pt")
REPO = Path("/kaggle/working/Image_Captioning")
COMMIT = "939369a"
OUTPUT = Path("/kaggle/working/caption_relation_supervision_audit")

assert "CHANGE_ME" not in str(SELECTED) and "CHANGE_ME" not in str(PILOT_REVIEW), "Dien SELECTED va PILOT_REVIEW."
for path in (SELECTED, COCO_JSON, TRAIN_GROUNDED_CACHE):
    assert path.is_file(), f"Missing file: {path}"
assert PILOT_REVIEW.exists(), f"Missing review ZIP/directory: {PILOT_REVIEW}"
assert not OUTPUT.exists(), "Doi OUTPUT sang ten moi neu chay lai."
if PILOT_REVIEW.is_dir():
    for arm in ("legacy", "direct"):
        assert (PILOT_REVIEW / arm / "evaluation_12" / "records.json").is_file(), "Chon folder chua legacy/ va direct/."

def run(args, cwd=None):
    args = list(map(str, args))
    print("Running:", " ".join(args), flush=True)
    subprocess.run(args, cwd=cwd, check=True)

if not REPO.exists():
    run(["git", "clone", "https://github.com/Supzxjee/Image_Captioning.git", REPO])
assert (REPO / ".git").is_dir(), REPO
run(["git", "fetch", "--depth", "30", "origin", "main"], REPO)
run(["git", "checkout", "--detach", COMMIT], REPO)
run([sys.executable, "-u", "audit_caption_relation_supervision.py", "--dataset-json", COCO_JSON, "--training-cache", TRAIN_GROUNDED_CACHE, "--manifest", REPO / "audit/decoder_adaptation_training_manifest.json", "--selected", SELECTED, "--pilot-review", PILOT_REVIEW, "--output", OUTPUT], REPO)
print("Download review:", str(OUTPUT) + "_review.zip", flush=True)
